<a href="https://colab.research.google.com/github/Rahat-Ferdause/Skill-Morph-Research-through-Data-Science/blob/Data_Preprocessing_and_Pipeline/Assignment_5_Data_Preprocessing_and_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Load the Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# TODO: Load the housing data
# df = pd.read_csv('Housing.csv')
from google.colab import drive
drive.mount('/content/drive')
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/Datasets/Housing/Housing.csv')  # No header in raw CSV

In [4]:
# TODO: How many houses are there?

# TODO: Show first 3 houses
# Your code here

print(f"Number of houses:")  # Fill this
print(df['furnishingstatus'].value_counts())


print("\nFirst 3 houses:")
print(df.head(3))

Number of houses:
furnishingstatus
semi-furnished    227
unfurnished       178
furnished         140
Name: count, dtype: int64

First 3 houses:
      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  


In [5]:
# Look at Columns
# TODO: Show all column names

print("Columns:")
# Your code here
print(df.columns)
# TODO: Show data types of columns
print("\nData types:")
print(df.dtypes)

Columns:
Index(['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'basement', 'hotwaterheating', 'airconditioning',
       'parking', 'prefarea', 'furnishingstatus'],
      dtype='object')

Data types:
price                int64
area                 int64
bedrooms             int64
bathrooms            int64
stories              int64
mainroad            object
guestroom           object
basement            object
hotwaterheating     object
airconditioning     object
parking              int64
prefarea            object
furnishingstatus    object
dtype: object


In [6]:
from sklearn.preprocessing import LabelEncoder
#Convert Yes/No to 1/0
# Check what's in our data
print("Current mainroad values:\n", df['mainroad'].head())

# Shows: yes, no, yes, yes, no


# Models need numbers, not text!
# We need: yes → 1, no → 0
label_encoder = LabelEncoder()

# Convert 'mainroad' column
df['mainroad_encoded'] = label_encoder.fit_transform(df['mainroad'])
print("\nAfter Label Encoding 'mainroad':")
print(df[['mainroad', 'mainroad_encoded']])
# no = 0, yes = 1

Current mainroad values:
 0    yes
1    yes
2    yes
3    yes
4    yes
Name: mainroad, dtype: object

After Label Encoding 'mainroad':
    mainroad  mainroad_encoded
0        yes                 1
1        yes                 1
2        yes                 1
3        yes                 1
4        yes                 1
..       ...               ...
540      yes                 1
541       no                 0
542      yes                 1
543       no                 0
544      yes                 1

[545 rows x 2 columns]


In [7]:
# Split into Train and Test
# TODO: Split - 80% train, 20% test
# Separate input (X) and output (y)
X = df.drop('furnishingstatus', axis=1)  # Everything except furnishingstatus
y = df['furnishingstatus']                # Only furnishingstatus

print("X has all Household information")
print("y has furnishingstatus")

# Split: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 20% for test
    random_state=20   # Same result every time
)

print(f"\nWe will train with: {len(X_train)} Households")
print(f"We will test with: {len(X_test)} Households")

X has all Household information
y has furnishingstatus

We will train with: 436 Households
We will test with: 109 Households


In [8]:
#Apply StandardScaler
# Area is huge (1650-13300), Bedrooms is small (1-6)!
#Your Code here
# Create scaler
scaler = StandardScaler()
# Columns to scale
cols_to_scale = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']

print("Before scaling:")
print(f"Area range: {X_train['area'].min():.0f} - {X_train['area'].max():.0f}")
print(f"Bedrooms range: {X_train['bedrooms'].min():.0f} - {X_train['bedrooms'].max():.0f}")
print()
# Learn from training data and scale it
X_train_scaled= scaler.fit_transform(X_train[cols_to_scale])

# Scale test data (just transform, don't fit)
X_test_scaled= scaler.transform(X_test[cols_to_scale])

print("After scaling:")
print(f"Area mean: {X_train['area'].mean():.2f}, std: {X_train['area'].std():.2f}")
print(f"Bedrooms mean: {X_train['bedrooms'].mean():.2f}, std: {X_train['bedrooms'].std():.2f}")
print()

# Show example values
print("Example - First row after scaling:")
print(X_train[cols_to_scale].iloc[0])
print()


Before scaling:
Area range: 1650 - 16200
Bedrooms range: 1 - 6

After scaling:
Area mean: 5155.65, std: 2186.35
Bedrooms mean: 2.97, std: 0.73

Example - First row after scaling:
area         4280
bedrooms        2
bathrooms       1
stories         1
parking         2
Name: 370, dtype: int64



In [9]:
#train and test model
#your code here
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# Check accuracy on training data
train_score = model.score(X_train_scaled, y_train)
print(f"Training accuracy: {train_score:.1%}")
# Check accuracy on test data
test_score = model.score(X_test_scaled, y_test)
print(f"Test accuracy: {test_score:.1%}")

Training accuracy: 43.1%
Test accuracy: 45.9%


In [10]:
from sklearn.model_selection import cross_val_score

# Do 5-fold cross-validation
scores = cross_val_score(
    model,           # Our model
    X_train_scaled,  # Training data
    y_train,         # Training labels
    cv=5            # 5 mini-tests
)

print("5 mini-test scores:")
for i in range(5):
    print(f"  Test {i+1}: {scores[i]:.1%}")

print(f"\nAverage: {scores.mean():.1%}")
print(f"This means our model is {scores.mean():.1%} accurate!")

5 mini-test scores:
  Test 1: 38.6%
  Test 2: 41.4%
  Test 3: 41.4%
  Test 4: 34.5%
  Test 5: 35.6%

Average: 38.3%
This means our model is 38.3% accurate!


In [11]:
# Predict a House Price
# New house details:
# area=5000, bedrooms=3, bathrooms=2, stories=2,
# mainroad=yes, guestroom=no, basement=yes,
# hotwaterheating=no, airconditioning=yes,
# parking=2, prefarea=yes, furnishingstatus=furnished
#your code here
import joblib

# Save the model
joblib.dump(model, 'my_model.pkl')
print("✅ Model saved as 'my_model.pkl'")

# Save the scaler
joblib.dump(scaler, 'my_scaler.pkl')
print("✅ Scaler saved as 'my_scaler.pkl'")

# Load saved model and scaler
model = joblib.load('my_model.pkl')
scaler = joblib.load('my_scaler.pkl')

# New patient data (8 features)
print("\nNew patient information:")
new_patient = [[5000, 3, 2, 2, 'yes', 'no', 0.201, 30]]
print("Pregnancies: 5")
print("Glucose: 116")
print("Blood Pressure: 74")
print("Age: 30")
print("(and other features...)")

# Scale the new patient data
new_patient_scaled = scaler.transform(new_patient)

# Make prediction
prediction = model.predict(new_patient_scaled)





✅ Model saved as 'my_model.pkl'
✅ Scaler saved as 'my_scaler.pkl'

New patient information:
Pregnancies: 5
Glucose: 116
Blood Pressure: 74
Age: 30
(and other features...)


ValueError: could not convert string to float: 'yes'

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# 1. LOAD YOUR DATA

# Assuming you have a housing dataset
df = df = pd.read_csv('/content/drive/MyDrive/Datasets/Housing/Housing.csv')

# Encode categorical variables
label = LabelEncoder()
for col in ['mainroad', 'guestroom', 'basement', 'hotwaterheating',
            'airconditioning', 'prefarea', 'furnishingstatus']:
    df[col] = label.fit_transform(df[col])

# Split features and target
X = df.drop('price', axis=1)
y = df['price']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Check accuracy on training data
train_score = model.score(X_train_scaled, y_train)
print(f"Training accuracy: {train_score:.1%}")

# Check accuracy on test data
test_score = model.score(X_test_scaled, y_test)
print(f"Test accuracy: {test_score:.1%}")


ValueError: X has 5 features, but LinearRegression is expecting 12 features as input.